In [ ]:
# pip install requests json python-binance pandas openpyxl tqdm
# python = 3.12

In [1]:
# 관련 docs
# [01]. binance api
#       https://developers.binance.com/docs/binance-spot-api-docs

In [33]:
import requests
import json
from binance.client import Client

import pandas as pd
import numpy as np
import openpyxl

import os
import time

from tqdm import tqdm

In [34]:
# 주의
KEYS = pd.read_excel("private_keys.xlsx")
api_key = KEYS['API Key'][0]
api_secret = KEYS['Secret Key'][0]

client = Client(api_key, api_secret)


In [45]:
KLINE_COLUMNS = [
    'Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 
    'Close time', 'Quote asset volume', 'Number of trades', 
    'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore'
]

class BinanceDataCollector:
    """
    바이낸스 API를 사용하여 심볼 정보 및 캔들스틱 데이터를 수집하는 클래스입니다.
    """
    def __init__(self, api_key, api_secret):
        # binance.client.Client 객체를 인스턴스 변수로 저장
        self.client = Client(api_key, api_secret)
        self.all_symbols = self.get_all_symbols()

    def get_all_symbols(self):
        """
        Binance 상장된 모든 Symbol 추출목적
        """
        info = self.client.get_exchange_info() # 심볼 + 필터
        
        symbols = [x['symbol'] for x in info['symbols']]
        print(f"전체 심볼 개수 : {len(symbols)}")
        return symbols

    def get_binance_klines(self, symbol, interval, start_str, end_str=None):
        # start_str과 end_str은 날짜 문자열 (예: '1 Jan, 2024').

        """
        Binance API에서 캔들스틱 데이터 추출
        , API 과다호출로 인한 ban을 받기 위해서 1회 호출시 symbol개수 한정
        """
        try:
            klines = self.client.get_historical_klines(
                symbol, 
                interval, 
                start_str, 
                end_str,
                limit = 2000 # 한 번에 500개 symbol만 호출
            )
            if not klines:
                print(f"{symbol} : 심볼 데이터 없음. 이유는 체크필요.")
                return None
            
            # DataFrame으로 변환하고 컬럼 이름 명시적으로 설정
            data = pd.DataFrame(klines, columns=KLINE_COLUMNS)

            # 데이터 타입 변환 및 인덱스 설정 (기존 get_binance_klines 로직 통합)
            data['Open time'] = pd.to_datetime(data['Open time'], unit='ms')
            data['Close time'] = pd.to_datetime(data['Close time'], unit='ms')
            
            numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in numeric_cols:
                # 숫자형 컬럼을 float으로 변환
                data[col] = pd.to_numeric(data[col])
                
            data = data.set_index('Open time')
            
            print(f"{symbol} : ({len(data)}개) 데이터 추출 완료")
            return data
        
        except Exception as e:
            print(f"{symbol} : 데이터 추출 오류 코드({e})")
            return None
        finally:
            sleep_time = np.random.randint(1,100+1)/100
            time.sleep(sleep_time) # ban 방지

In [46]:
data_collector = BinanceDataCollector(api_key, api_secret)

# INTERVAL = Client.KLINE_INTERVAL_1HOUR
INTERVAL = Client.KLINE_INTERVAL_1DAY

START_DATE = "1 May, 2024"

전체 심볼 개수 : 3549


In [47]:
# 데이터 dict형식으로 받을 예정
all_klines_data = {}

print("==========Data Extract Process is started.==========")

df_list =[]

# 인스턴스 변수에 저장된 전체 심볼 목록 사용
all_symbols = data_collector.all_symbols
all_symbols = [x for x in all_symbols]
all_symbols = [x for x in all_symbols if x.endswith('USDT')]
print(len(all_symbols)) # 657

# test
# for symbol in tqdm(all_symbols,miniters=100, desc="처리 중"):
for symbol in tqdm(all_symbols,miniters=100, desc="처리 중"):
    df_klines = data_collector.get_binance_klines(symbol, INTERVAL, START_DATE)
    
    if df_klines is not None:
        df_klines['Symbol'] = symbol
        # all_klines_data[symbol] = df_klines
        df_list.append(df_klines)

if df_list:
    df_final = pd.concat(df_list)
    df_final = df_final.sort_values(by = ['Symbol', 'Open time'])
    print("==========Data Extract Process is done.==========")
    print(f"총 수집된 행(Row) 수: {len(df_final)}")
    print(df_final.head())
else :
    print('no data')

==========Data Extract Process is started.==========
657


처리 중:   0%|          | 0/657 [00:00<?, ?it/s]

BTCUSDT : (699개) 데이터 추출 완료
ETHUSDT : (699개) 데이터 추출 완료
BNBUSDT : (699개) 데이터 추출 완료
BCCUSDT : 심볼 데이터 없음. 이유는 체크필요.
NEOUSDT : (699개) 데이터 추출 완료
LTCUSDT : (699개) 데이터 추출 완료
QTUMUSDT : (699개) 데이터 추출 완료
ADAUSDT : (699개) 데이터 추출 완료
XRPUSDT : (699개) 데이터 추출 완료
EOSUSDT : (391개) 데이터 추출 완료
TUSDUSDT : (699개) 데이터 추출 완료
IOTAUSDT : (699개) 데이터 추출 완료
XLMUSDT : (699개) 데이터 추출 완료
ONTUSDT : (699개) 데이터 추출 완료
TRXUSDT : (699개) 데이터 추출 완료
ETCUSDT : (699개) 데이터 추출 완료
ICXUSDT : (699개) 데이터 추출 완료
VENUSDT : 심볼 데이터 없음. 이유는 체크필요.
NULSUSDT : (351개) 데이터 추출 완료
VETUSDT : (699개) 데이터 추출 완료
PAXUSDT : 심볼 데이터 없음. 이유는 체크필요.
BCHABCUSDT : 심볼 데이터 없음. 이유는 체크필요.
BCHSVUSDT : 심볼 데이터 없음. 이유는 체크필요.
USDCUSDT : (699개) 데이터 추출 완료
LINKUSDT : (699개) 데이터 추출 완료


처리 중:   4%|▍         | 25/657 [00:18<07:53,  1.34it/s]

WAVESUSDT : (48개) 데이터 추출 완료


...

KATUSDT : (13개) 데이터 추출 완료


처리 중: 100%|█████████▉| 656/657 [07:11<00:00,  1.96it/s]

XAUTUSDT : (5개) 데이터 추출 완료


처리 중: 100%|██████████| 657/657 [07:12<00:00,  1.52it/s]


==========Data Extract Process is done.==========
총 수집된 행(Row) 수: 283224
             Open   High    Low  Close       Volume              Close time  \
Open time                                                                     
2025-09-22  1.000  7.260  1.000  4.839  69411652.78 2025-09-22 23:59:59.999   
2025-09-23  4.838  7.180  4.566  5.815  52749822.84 2025-09-23 23:59:59.999   
2025-09-24  5.820  5.895  4.838  5.005  19526670.81 2025-09-24 23:59:59.999   
2025-09-25  5.006  5.014  3.649  3.917  13281842.95 2025-09-25 23:59:59.999   
2025-09-26  3.917  4.455  3.341  3.679  14793109.26 2025-09-26 23:59:59.999   

            Quote asset volume  Number of trades Taker buy base asset volume  \
Open time                                                                      
2025-09-22  343688450.68410000           1204307           34992716.27000000   
2025-09-23  304937872.83330000           1340566           26663358.92000000   
2025-09-24  102258777.96859000            512620     

In [10]:
len(df_list)

250

In [12]:
df_all = pd.DataFrame()
for df_tmp in df_list:
    df_all = pd.concat([df_all, df_tmp],ignore_index=True)

In [19]:
import pyarrow

In [23]:
df_all.to_parquet('./data/df_260315.parquet',engine='fastparquet')